# 01 — Crash Frequency Model Search

This template runs a **mixed Negative Binomial crash-frequency model search**
on the Example 16-3 road-segment dataset.  The search explores which
variables to include, whether each coefficient should be fixed or random,
and whether the count family should be Poisson or Negative Binomial.

**Workflow**
1. Load the bundled dataset
2. Create an exposure (VMT) offset
3. Declare structural constraints with `ModelConstraints`
4. Run a simulated-annealing structure search
5. Re-fit the best model and inspect results

## 1 · Installation

```bash
pip install metacountregressor
```

In [ ]:
import numpy as np
import pandas as pd

from metacountregressor import (
    ExperimentBuilder,
    ModelConstraints,
    load_example16_3_model_data,
    get_help,
)

# Optional: print the full crash-frequency usage guide
# get_help('crash_frequency')

## 2 · Load data

In [ ]:
df = load_example16_3_model_data()
print(df.shape)
df.head(3)

In [ ]:
# Exposure offset: vehicle-miles travelled (scaled to 10^-8)
exposure = df['LENGTH'] * df['AADT'] * 365 / 1e8
df['OFFSET'] = np.log(exposure.clip(lower=1e-9))

print('Outcome (FREQ) summary:')
print(df['FREQ'].describe())
print('\nOffset summary:')
print(df['OFFSET'].describe())

## 3 · Select variables and declare constraints

**Role codes quick reference**
| Code | Role |
|------|------|
| 0 | Excluded |
| 1 | Fixed |
| 2 | Random (independent) |
| 3 | Random (correlated) |
| 4 | Grouped |
| 5 | Heterogeneity in means |
| 6 | Zero inflation |

Use `ModelConstraints` to restrict which roles and distributions each
variable may take.  All methods return `self` for chaining.

In [ ]:
# Variables to include in the search (edit to your dataset)
variables = [
    'URB', 'ACCESS', 'GRADEBR', 'CURVES', 'LENGTH',
    'SPEED', 'WIDTH', 'SLOPE', 'AVEPRE',
]

c = (
    ModelConstraints()
    # OFFSET must always be in the model (exposure term)
    .force_include('OFFSET')
    # Geometric road features are not plausible zero-inflation terms
    .no_zi('LENGTH', 'CURVES', 'GRADEBR', 'WIDTH', 'SLOPE')
    # Allow curvature to be a random parameter (lognormal = positive effect)
    .allow_random('CURVES', distributions=['lognormal'])
    # Binary indicator — no taste variation
    .no_random('URB')
)

# Inspect resolved constraints
print(c)

## 4 · Build the experiment and explore the data

In [ ]:
builder = ExperimentBuilder(
    df=df,
    id_col='ID',
    y_col='FREQ',
    offset_col='OFFSET',
)

# Summary of the data and variable roles
builder.describe()

In [ ]:
evaluator = builder.build_evaluator(
    variables=variables,
    constraints=c,
    # default_roles: which roles any variable can take by default
    # [0,1,2,3,5] = excluded / fixed / random-ind / random-cor / hetro
    default_roles=[0, 1, 2, 3, 5],
    max_latent_classes=1,   # single class for basic crash freq search
    mode='single',          # minimise BIC
    R=200,                  # Halton simulation draws
)

## 5 · Run the search

In [ ]:
result = builder.run(
    evaluator=evaluator,
    algo='sa',        # simulated annealing  |  'de' = differential evolution
    max_iter=50,      # increase to 200+ for a thorough search
    seed=42,
)

print('Best BIC:', result.best_score)
print('Best solution:', result.best_solution)

## 6 · Re-fit and inspect the best model

In [ ]:
# Re-estimate the winning structure with more draws for accurate SEs
fit = builder.fit_manual_model(
    manual_spec=result.best_spec,
    model='nb',
    R=500,
)
print(fit)

## 7 · Next steps

- **Latent classes**: see `02_latent_class_fc_validation.ipynb`
- **CMF model**: see `03_cmf_aadt_search.ipynb`
- **Linear model**: see `04_linear_speed_prediction.ipynb`
- **More help**: `get_help('crash_frequency')`, `get_help('constraints')`